# Note 7: Next Steps — What Advanced Solvers Do Differently

We've built a complete mental model from naive Gaussian elimination to multifrontal sparse factorization. This is the foundation that every production solver builds on. But state-of-the-art solvers add layers of sophistication on top. This note surveys where they go further and where open problems remain.

## What We've Covered vs. What Production Solvers Do

| What we built | What production solvers add |
|---|---|
| LDL^T with Bunch-Kaufman | Threshold pivoting (trade accuracy for sparsity) |
| Exact minimum degree ordering | Approximate minimum degree (AMD), nested dissection |
| Simple supernodes | Relaxed supernodes, supernode amalgamation |
| Sequential tree traversal | Parallel level-set scheduling, task DAGs |
| One-shot factorization | Scaling/equilibration as preprocessing |
| Basic iterative refinement | Adaptive refinement with convergence monitoring |

## 1. Threshold Pivoting

In Note 3, Bunch-Kaufman pivoting always chooses the numerically best pivot. This is optimal for stability but causes extra fill-in because pivoting disrupts the predicted sparsity pattern from symbolic analysis.

**Threshold pivoting** relaxes this: accept any pivot that's "good enough" (within a threshold of the best), preferring the one that preserves sparsity.

$$\text{Accept pivot } a_{kk} \text{ if } |a_{kk}| \geq \tau \cdot \max_i |a_{ik}|$$

where $\tau \in (0, 1]$ is the threshold. MUMPS defaults to $\tau = 0.01$; rmumps uses the same.

- $\tau = 1.0$: full partial pivoting (most stable, most fill)
- $\tau = 0.01$: very permissive (less stable, much less fill)
- $\tau = 0.0$: no pivoting at all (risky, but zero extra fill)

The iterative refinement step (Note 6) compensates for the reduced pivot quality. This is a key trade-off in sparse solvers: **stability vs. sparsity**.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# Demonstrate the stability-sparsity tradeoff
def ldlt_threshold(A, threshold=0.01):
    """LDL^T with threshold pivoting (1x1 pivots only for simplicity).
    
    Accept the diagonal pivot if |a_kk| >= threshold * max|a_ik|.
    Otherwise swap rows to get a better pivot.
    """
    n = A.shape[0]
    A = A.astype(float).copy()
    L = np.eye(n)
    d = np.zeros(n)
    perm = list(range(n))
    swaps = 0
    
    for k in range(n):
        # Check if diagonal pivot is acceptable
        col_max = max(abs(A[i, k]) for i in range(k, n)) if k < n else abs(A[k, k])
        
        if col_max > 1e-15 and abs(A[k, k]) < threshold * col_max:
            # Need to swap — find the best pivot
            best = k + np.argmax([abs(A[i, k]) for i in range(k, n)])
            if best != k:
                A[[k, best]] = A[[best, k]]
                A[:, [k, best]] = A[:, [best, k]]
                L[[k, best], :k] = L[[best, k], :k]
                perm[k], perm[best] = perm[best], perm[k]
                swaps += 1
        
        d[k] = A[k, k]
        if abs(d[k]) < 1e-15:
            continue
        
        for i in range(k + 1, n):
            L[i, k] = A[i, k] / d[k]
            for j in range(k + 1, i + 1):
                A[i, j] -= L[i, k] * d[k] * L[j, k]
                A[j, i] = A[i, j]
    
    return L, d, perm, swaps


# Test on a matrix where pivoting matters
n = 10
np.random.seed(42)
B = np.random.randn(n, n)
A = B @ B.T + 0.1 * np.eye(n)  # SPD but some small pivots

for tau in [1.0, 0.1, 0.01, 0.001]:
    L, d, perm, swaps = ldlt_threshold(A, threshold=tau)
    D = np.diag(d)
    P = np.eye(n)[perm]
    recon = L @ D @ L.T
    err = np.linalg.norm(P @ A @ P.T - recon) / np.linalg.norm(A)
    nnz_L = np.sum(np.abs(L) > 1e-12)
    print(f"tau={tau:.3f}  swaps={swaps:2d}  nnz(L)={nnz_L:3d}  reconstruction err={err:.2e}")

tau=1.000  swaps= 1  nnz(L)= 55  reconstruction err=1.22e-16
tau=0.100  swaps= 0  nnz(L)= 55  reconstruction err=1.35e-16
tau=0.010  swaps= 0  nnz(L)= 55  reconstruction err=1.35e-16
tau=0.001  swaps= 0  nnz(L)= 55  reconstruction err=1.35e-16


## 2. Nested Dissection Ordering

AMD (Note 5) is a **local** greedy heuristic — it makes the best choice at each step. **Nested dissection** is a **global** strategy based on graph partitioning.

The idea:
1. Find a small set of nodes (a **separator**) whose removal splits the graph into two roughly equal parts
2. Number the separator nodes **last**
3. Recursively apply to each part

For a 2D problem on an $m \times m$ grid:
- AMD fill: $O(n \log n)$ where $n = m^2$
- Nested dissection fill: $O(n \log n)$ (same asymptotically, but smaller constants)

For 3D problems, nested dissection wins decisively: $O(n^{4/3})$ vs $O(n^2)$ fill.

Libraries like **METIS** and **Scotch** implement this. rmumps currently uses only AMD — adding nested dissection for 3D problems would be a significant improvement.

In [2]:
# Demonstrate nested dissection on a small 1D example
# (where it's easy to see the separator structure)

def tridiagonal(n):
    """n x n tridiagonal matrix (1D Laplacian)."""
    A = np.zeros((n, n))
    for i in range(n):
        A[i, i] = 2.0
        if i > 0: A[i, i-1] = -1.0
        if i < n-1: A[i, i+1] = -1.0
    return A

def nested_dissection_1d(n):
    """Compute a nested dissection ordering for a 1D chain of n nodes."""
    perm = []
    
    def nd_recurse(start, end):
        if start > end:
            return
        if start == end:
            perm.append(start)
            return
        
        mid = (start + end) // 2
        # Recurse on left and right parts FIRST
        nd_recurse(start, mid - 1)
        nd_recurse(mid + 1, end)
        # Separator (midpoint) LAST
        perm.append(mid)
    
    nd_recurse(0, n - 1)
    return perm


n = 15
A = tridiagonal(n)
perm_nd = nested_dissection_1d(n)
perm_natural = list(range(n))

# Compare fill-in
L_nat = np.linalg.cholesky(A)

P = np.eye(n)[perm_nd]
L_nd = np.linalg.cholesky(P @ A @ P.T)

nnz_nat = np.sum(np.abs(L_nat) > 1e-12)
nnz_nd = np.sum(np.abs(L_nd) > 1e-12)

print(f"1D tridiagonal, n={n}")
print(f"Natural ordering: nnz(L) = {nnz_nat}")
print(f"Nested dissection: nnz(L) = {nnz_nd}")
print(f"\nND permutation: {perm_nd}")
print("(separators — the midpoints — appear last)")

1D tridiagonal, n=15
Natural ordering: nnz(L) = 29
Nested dissection: nnz(L) = 37

ND permutation: [0, 2, 1, 4, 6, 5, 3, 8, 10, 9, 12, 14, 13, 11, 7]
(separators — the midpoints — appear last)


## 3. Scaling and Equilibration

Before factoring, production solvers **scale** the matrix to improve numerical behavior. The goal: make all entries roughly the same magnitude.

**Diagonal scaling:** $\hat{A} = D_s A D_s$ where $D_s$ is diagonal.

**Ruiz iterative scaling** (used by rmumps):
1. For each row/column, compute the maximum absolute value
2. Scale so each row/column has max entry = 1
3. Repeat for several iterations

This is cheap ($O(\text{nnz})$ per iteration) and dramatically improves accuracy for poorly scaled problems.

In [3]:
def ruiz_scaling(A, max_iters=10):
    """Ruiz iterative equilibration for a symmetric matrix.
    
    Returns scaled matrix and scaling vector.
    """
    n = A.shape[0]
    A_scaled = A.astype(float).copy()
    s = np.ones(n)  # cumulative scaling
    
    for iteration in range(max_iters):
        # Compute row/col max (same for symmetric)
        d = np.zeros(n)
        for i in range(n):
            d[i] = np.max(np.abs(A_scaled[i, :]))
        
        if np.min(d) < 1e-15:
            break
        
        d = 1.0 / np.sqrt(d)
        
        # Apply: A_scaled = diag(d) @ A_scaled @ diag(d)
        A_scaled = np.diag(d) @ A_scaled @ np.diag(d)
        s *= d
        
        # Check convergence: all row/col maxima close to 1?
        row_max = np.max(np.abs(A_scaled), axis=1)
        if np.max(np.abs(row_max - 1.0)) < 0.01:
            break
    
    return A_scaled, s


# Badly scaled matrix
n = 5
A_base = np.array([
    [1e6, 1.0, 0, 0, 0],
    [1.0, 1e-6, 1.0, 0, 0],
    [0, 1.0, 1.0, 1.0, 0],
    [0, 0, 1.0, 1e4, 1.0],
    [0, 0, 0, 1.0, 1e-4]
])
A_base = A_base + A_base.T  # symmetrize
np.fill_diagonal(A_base, np.diag(A_base) / 2)

A_scaled, s = ruiz_scaling(A_base)

print("Original diagonal:")
print(f"  {np.diag(A_base)}")
print(f"  Ratio max/min: {np.max(np.abs(np.diag(A_base))) / np.min(np.abs(np.diag(A_base))):.0e}")

print(f"\nScaled diagonal:")
print(f"  {np.diag(A_scaled)}")
print(f"  Ratio max/min: {np.max(np.abs(np.diag(A_scaled))) / np.min(np.abs(np.diag(A_scaled))):.1f}")

print(f"\nCondition number: {np.linalg.cond(A_base):.1e} → {np.linalg.cond(A_scaled):.1e}")

Original diagonal:
  [1000000.           0.           1.       10000.           0.0001]
  Ratio max/min: 1e+12

Scaled diagonal:
  [1.     0.     0.5    1.     0.2459]
  Ratio max/min: 2000000.0

Condition number: 3.3e+09 → 3.8e+00


## 4. Supernode Amalgamation

In Note 5, we defined supernodes as consecutive columns with **identical** sparsity patterns. In practice, solvers **relax** this criterion:

- **Relaxed supernodes:** Allow a small amount of extra fill ("artificial zeros") to merge columns into larger supernodes
- **Benefit:** Larger dense blocks → better BLAS utilization → faster on modern hardware
- **Cost:** Some wasted flops on the artificial zeros

The trade-off is almost always worth it. A $32 \times 32$ dense block with 20% artificial zeros is faster than four $8 \times 16$ blocks with no waste, because BLAS matrix-multiply throughput scales superlinearly with block size due to cache effects.

rmumps implements basic supernodal grouping. More aggressive amalgamation (like CHOLMOD or PARDISO) could further improve performance on large problems.

## 5. Mixed Precision and Communication-Avoiding Algorithms

These are active research frontiers:

### Mixed Precision
- Factor in **single precision** (float32) — 2x faster, half the memory
- Refine in **double precision** (float64) — iterative refinement recovers full accuracy
- The factorization is the bottleneck; refinement is cheap
- Can deliver double-precision accuracy at nearly single-precision speed

### Communication-Avoiding
- On modern hardware, **memory bandwidth** is the bottleneck, not arithmetic
- Reorganize algorithms to minimize data movement between cache levels
- "Tall-skinny" frontal matrices can be processed with communication-optimal algorithms
- Relevant for both shared-memory (cache) and distributed-memory (network) parallelism

## 6. GPU Acceleration

GPUs excel at dense matrix operations — exactly what the multifrontal method uses for frontal matrices.

The strategy:
- **Small fronts** (near the leaves): process on CPU (GPU kernel launch overhead dominates)
- **Large fronts** (near the root): offload to GPU (dense BLAS dominates)
- The root front is often the single most expensive operation

Solvers like CHOLMOD (SuiteSparse) and cuSOLVER implement this. rmumps is currently CPU-only — GPU offloading of large fronts is a clear opportunity.

### Estimated Impact

For 3D problems with large fronts (e.g., $n > 50{,}000$):
- The root front might be $1000 \times 1000$ dense
- GPU dense factorization: 10-50x faster than CPU
- But the root front is only one of many supernodes, so overall speedup is limited by Amdahl's law
- Typical end-to-end speedup: 2-5x on suitable problems

## 7. Iterative Solvers and Preconditioning

Everything in Notes 1-6 is about **direct solvers** — they compute an exact factorization (up to rounding). The alternative is **iterative solvers** (conjugate gradient, GMRES, etc.) that compute an approximate solution by matrix-vector products only.

| | Direct | Iterative |
|---|---|---|
| Cost | $O(\text{nnz}(L))$ factor + $O(\text{nnz}(L))$ solve | $O(k \cdot \text{nnz}(A))$ for $k$ iterations |
| Memory | Must store $L$ (can be much larger than $A$) | Only need $A$ and a few vectors |
| Robustness | Always works (up to singularity) | May not converge, needs good preconditioner |
| Best for | Stiff problems, indefinite systems, repeated solves | Very large problems, good preconditioners available |

For interior point methods, direct solvers are standard because:
1. The KKT system is **indefinite** (hard for iterative methods)
2. We solve **many systems** with the same pattern (amortize symbolic analysis)
3. We need **high accuracy** in the Newton step

However, for very large problems ($n > 10^6$), memory for $L$ becomes prohibitive, and iterative methods with **incomplete factorization preconditioners** become attractive. This is an active research area.

### Hybrid Approaches

Use the multifrontal framework to build a **preconditioner** rather than an exact factorization:
- Drop small entries during factorization (incomplete LDL^T)
- Use the approximate factor as a preconditioner for an iterative solver
- Gives the robustness of direct methods with the memory scaling of iterative methods

## 8. Where rmumps Stands and Where It Could Go

### Current Strengths

- **Pure Rust, memory-safe:** No C/Fortran dependencies, no segfaults
- **Full Bunch-Kaufman with inertia:** Handles indefinite KKT systems correctly
- **Cached symbolic analysis:** Fast re-factorization across IPM iterations
- **Adaptive iterative refinement:** Compensates for threshold pivoting
- **Parallel supernodal factorization:** Uses rayon for level-set parallelism
- **100% Hock-Schittkowski, 82% CUTEst:** Competitive with IPOPT/MA57

### Concrete Improvement Opportunities

| Improvement | Expected Impact | Difficulty |
|---|---|---|
| Nested dissection ordering (METIS) | Better fill for 3D problems | Medium |
| Supernode amalgamation | Larger BLAS blocks, better cache use | Medium |
| Mixed precision (f32 factor + f64 refine) | ~2x faster factorization | Low-Medium |
| GPU offloading of large fronts | 2-5x for large 3D problems | High |
| Block-level parallelism in frontal ops | Better scaling beyond 4 cores | Medium |
| Out-of-core factorization | Handle problems larger than RAM | High |
| SIMD-optimized dense kernels | 2-4x for frontal matrix ops | Medium |
| Incomplete factorization preconditioner | Enable iterative solve for huge problems | High |

## 9. The Broader Landscape

### Major Sparse Direct Solver Libraries

| Solver | Language | Pivoting | Ordering | GPU | Notes |
|---|---|---|---|---|---|
| **MA57/MA97** (HSL) | Fortran | Bunch-Kaufman | AMD, METIS | No | IPOPT's default; gold standard for IPM |
| **MUMPS** | Fortran | Threshold BK | AMD, METIS, Scotch | Yes | Distributed memory, very general |
| **PARDISO** (MKL) | C/Fortran | BK + threshold | METIS | No | Highly optimized for Intel CPUs |
| **CHOLMOD** (SuiteSparse) | C | Supernodal Cholesky | AMD, METIS | Yes | Best for SPD systems |
| **SuperLU** | C | Partial pivoting | COLAMD | Yes | General (non-symmetric) |
| **rmumps** | Rust | Bunch-Kaufman | AMD | No | Pure Rust, memory-safe, this project |

### What Makes MA57 Hard to Beat

MA57 (from the Harwell Subroutine Library) is the default linear solver in IPOPT and has been refined over 30+ years. Its advantages:
1. Extremely tuned Fortran inner loops
2. Sophisticated pivot selection that balances fill and stability
3. Memory-efficient frontal matrix management
4. Battle-tested on thousands of optimization problems

rmumps achieves competitive results through a different trade-off: slightly more fill-in and computation, but complete memory safety and no external dependencies.

## 10. Open Research Problems

Some genuinely open questions in sparse linear algebra:

1. **Optimal ordering is NP-hard.** AMD and nested dissection are heuristics. Can machine learning find better orderings for specific problem classes?

2. **Adaptivity during factorization.** Can we dynamically adjust the ordering as we discover the actual fill pattern (not just the predicted one)?

3. **Mixed direct-iterative methods.** How to best combine exact direct solves on subproblems with iterative methods for the global problem?

4. **Randomized numerical linear algebra.** Sketching and random sampling can approximate factorizations in $O(n \log n)$ for structured matrices. Can this help for KKT systems?

5. **Hardware-aware algorithms.** Modern CPUs, GPUs, and accelerators have complex memory hierarchies. How should sparse factorization algorithms co-design with hardware?

6. **Certified numerics.** Can we provide guaranteed error bounds for sparse factorization, not just empirical residual checks?

## Summary: The Complete Map

```
Note 1: Naive GE          → "It works but is fragile"
  |
  | (fix stability)
  v
Note 2: Pivoting + LU     → "Stable and separates factor from solve"
  |
  | (exploit symmetry)
  v
Note 3: LDL^T + BK        → "Half the work, inertia for free"
  |
  | (break the n^2 barrier)
  v
Note 4: Sparse formats     → "Store and compute with only nonzeros"
  |
  | (control fill-in)
  v
Note 5: AMD + etree        → "Predict structure, minimize fill"
  |
  | (make it fast)
  v
Note 6: Multifrontal       → "Dense BLAS on small blocks, tree parallelism"
  |
  | (push the frontier)
  v
Note 7: Next steps         → Threshold pivoting, nested dissection,
                              GPU, mixed precision, iterative hybrids
```

You now have the conceptual framework to read and understand the source code of any production sparse solver — including rmumps.

---

*This is Note 7, the final note in a series building from basic Gaussian elimination to the multifrontal sparse solver used in [ripopt](https://github.com/jkitchin/ripopt).*